# YOLO12-Small + EMA-32 — ARDD26

Revisi dari notebook baseline ARDD26. Notebook ini meng-clone branch `yolo12-ema32`, memakai `yolo12-ema32.yaml`, mentransfer bobot YOLO12s yang kompatibel secara indeks, lalu menjalankan train → validation → test → ZIP. Efficient Multi-scale Attention (EMA factor 32) ditempatkan pada P3/8; GhostConv tidak digunakan.

Aktifkan **GPU** dan **Internet** pada Kaggle sebelum menjalankan notebook.

In [ ]:
# 1. Clone source arsitektur EMA-32 dan siapkan environment Kaggle.
import json
import platform
import re
import subprocess
import sys
import warnings
import zipfile
from collections import Counter
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-ema32'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022-ema32'

def log_section(title):
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

def run(command):
    subprocess.run(command, check=True)

log_section('CLONE AND INSTALL YOLO12S + EMA-32')
if REPO_DIR.exists():
    run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])
else:
    run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(
    f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8'
)
warnings.filterwarnings(
    'ignore', message=r'`get_trial_id` is meant to only be called.*',
    category=UserWarning, module=r'ray\.train\._internal\.session',
)
sys.path.insert(0, str(REPO_DIR))
import torch
import ultralytics
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Branch      : {REPO_BRANCH}')
print(f'Commit      : {REPO_COMMIT}')
print(f'Device      : {DEVICE}')
print(f'GPU         : {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'WARNING: GPU tidak aktif')


In [ ]:
# 2. Audit dataset, bangun model EMA-32, dan transfer pretrained YOLO12s.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ardd26/datasets_here')
DATA_YAML = WORKDIR / 'ardd26_4class.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-ema32.yaml'
CLASS_NAMES = ['D00', 'D10', 'D20', 'D40']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
EMA_FACTOR = 32
CUSTOM_SOURCE_FILES = (
    REPO_DIR / 'ultralytics/nn/modules/conv.py',
    REPO_DIR / 'ultralytics/nn/modules/__init__.py',
    REPO_DIR / 'ultralytics/nn/tasks.py',
    MODEL_YAML,
)

# Sama dengan baseline: physical batch 16 dan nominal batch 64 untuk gradient accumulation.
EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED = 0, 2, 42
EXPERIMENT_NAME = 'yolo12s_ema32_pretrained_ardd26'
RUNS_DIR = WORKDIR / 'runs'

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset tidak ditemukan: {DATA_ROOT}')
if not MODEL_YAML.is_file():
    raise FileNotFoundError(f'Model YAML tidak ditemukan: {MODEL_YAML}')

def file_stems(folder, extensions):
    if not folder.is_dir():
        raise FileNotFoundError(f'Folder tidak ditemukan: {folder}')
    return {path.stem.casefold() for path in folder.iterdir() if path.is_file() and path.suffix.lower() in extensions}

dataset_report = {}
for split in ('train', 'val', 'test'):
    image_dir, label_dir = (DATA_ROOT / split / name for name in ('images', 'labels'))
    image_stems = file_stems(image_dir, IMAGE_EXTENSIONS)
    label_stems = file_stems(label_dir, {'.txt'})
    if image_stems != label_stems:
        raise ValueError(f'Pairing {split} gagal: images={len(image_stems)}, labels={len(label_stems)}')
    counts, empty_labels = Counter(), 0
    for label_path in sorted(label_dir.glob('*.txt')):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        empty_labels += int(not lines)
        for line_number, line in enumerate(lines, 1):
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f'{label_path}:{line_number}: format bukan 5 kolom')
            class_id, coordinates = int(parts[0]), [float(value) for value in parts[1:]]
            if class_id not in range(len(CLASS_NAMES)):
                raise ValueError(f'{label_path}:{line_number}: class ID di luar rentang')
            if not all(0 <= value <= 1 for value in coordinates) or coordinates[2] <= 0 or coordinates[3] <= 0:
                raise ValueError(f'{label_path}:{line_number}: bounding box invalid')
            counts[class_id] += 1
    dataset_report[split] = {
        'pairs': len(image_stems), 'objects': sum(counts.values()), 'empty_labels': empty_labels,
        'classes': {CLASS_NAMES[index]: counts[index] for index in range(len(CLASS_NAMES))},
    }

DATA_YAML.write_text(
    f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40
''',
    encoding='utf-8',
)
log_section('DATASET AUDIT PASSED')
print(json.dumps(dataset_report, indent=2))

from ultralytics.nn.modules import EMAAttention, GhostConv
from ultralytics.nn.tasks import DetectionModel

log_section('YOLO12S + EMA-32 MODEL INFO')
print(MODEL_YAML.read_text(encoding='utf-8'))
check_model = DetectionModel(str(MODEL_YAML), nc=4, verbose=True)
ema_layers = [(layer.i, layer.groups) for layer in check_model.model if isinstance(layer, EMAAttention)]
ghost_layers = [layer.i for layer in check_model.model if isinstance(layer, GhostConv)]
assert ema_layers == [(5, EMA_FACTOR)], f'EMA-32 tidak sesuai YAML: {ema_layers}'
assert not ghost_layers, f'Varian EMA murni tidak boleh memuat GhostConv: {ghost_layers}'
print(f'Parameters (4 classes): {sum(parameter.numel() for parameter in check_model.parameters()):,}')
print(f'EMA layers           : {ema_layers}')
del check_model

def target_layer_index(source_index):
    return source_index + int(source_index >= 5)

def remap_yolo12s_weights(source_state, target_state):
    transferred = {}
    pattern = re.compile(r'^model\.(\d+)(\..+)$')
    for source_key, source_tensor in source_state.items():
        match = pattern.match(source_key)
        if match is None:
            continue
        target_key = f'model.{target_layer_index(int(match.group(1)))}{match.group(2)}'
        if target_key in target_state and target_state[target_key].shape == source_tensor.shape:
            transferred[target_key] = source_tensor
    return transferred

log_section('PRETRAINED WEIGHT TRANSFER')
model = YOLO(str(MODEL_YAML))
source_model = YOLO('yolo12s.pt').model.float()
target_state = model.model.state_dict()
transferred_state = remap_yolo12s_weights(source_model.state_dict(), target_state)
incompatible = model.model.load_state_dict(transferred_state, strict=False)
PRETRAINED_REPORT = {
    'source_weights': 'yolo12s.pt',
    'policy': 'remap compatible YOLO12s tensors around EMA insertion after source layer 4',
    'transferred_tensors': len(transferred_state),
    'target_tensors': len(target_state),
    'uninitialized_tensors': len(incompatible.missing_keys),
    'uninitialized_tensor_names': incompatible.missing_keys,
}
model.ckpt = {'model': model.model}
print(f'Transferred tensors : {PRETRAINED_REPORT["transferred_tensors"]}/{PRETRAINED_REPORT["target_tensors"]}')
print(f'Uninitialized tensors: {PRETRAINED_REPORT["uninitialized_tensors"]} (EMA-32 dan head 4 kelas)')
del source_model, target_state, transferred_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 3. Fine-tuning, kemudian validation dan test memakai best.pt.
log_section('TRAINING — YOLO12S + EMA-32')
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
    device=DEVICE, workers=WORKERS, project=str(RUNS_DIR), name=EXPERIMENT_NAME,
    exist_ok=False, pretrained=True, optimizer=OPTIMIZER, lr0=LR0, momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY, cos_lr=False, patience=PATIENCE, seed=SEED,
    deterministic=True, plots=True, verbose=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT, LAST_PT = Path(model.trainer.best), Path(model.trainer.last)
if not BEST_PT.is_file():
    raise FileNotFoundError(f'best.pt tidak ditemukan: {BEST_PT}')

def metric_summary(metrics):
    return {
        'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
        'map50_95_per_class': {CLASS_NAMES[index]: float(value) for index, value in enumerate(metrics.box.maps)},
        'save_dir': str(metrics.save_dir),
    }

best_model = YOLO(str(BEST_PT))
reports = {}
for split in ('val', 'test'):
    log_section(f'{split.upper()} — BEST.PT')
    metrics = best_model.val(
        data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
        project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_{split}', exist_ok=False, plots=True,
    )
    reports[split] = metric_summary(metrics)
    print(json.dumps(reports[split], indent=2))
VAL_REPORT, TEST_REPORT = reports['val'], reports['test']


In [ ]:
# 4. Simpan metadata reproduksibilitas, checkpoint, plot, dan metrik dalam satu ZIP.
EVALUATION_REPORT = {'validation': VAL_REPORT, 'test': TEST_REPORT}
EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'repository_url': REPO_URL, 'repository_branch': REPO_BRANCH, 'repository_commit': REPO_COMMIT,
    'model_yaml': str(MODEL_YAML), 'architecture': 'YOLO12s + EMA-32 at P3/8', 'ema_factor': EMA_FACTOR,
    'dataset_root': str(DATA_ROOT), 'classes': CLASS_NAMES, 'dataset_report': dataset_report,
    'pretrained_transfer': PRETRAINED_REPORT, 'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS,
    'optimizer': OPTIMIZER, 'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive, path):
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    file_count = sum(add_to_zip(archive, item) for item in (
        RUN_DIR, Path(VAL_REPORT['save_dir']), Path(TEST_REPORT['save_dir']), DATA_YAML,
        *CUSTOM_SOURCE_FILES, REPO_METADATA, RUN_CONFIG, EVALUATION_JSON,
    ))

log_section('EXPERIMENT COMPLETED')
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created: {ZIP_PATH} ({file_count} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
